# Задачи:
### PCA:
- ~~Написать функцию центрирования матриц~~
- ~~Реализовать QR алгоритм для нахождения собственных векторов и чисел~~
- Посмотреть какие еще есть алгоритмы сингулярного разложения, у которых меньше сложность (дополнительно) **randomized truncated SVD**
- Найти датасет, заскелить, провести PCA и обучить модель на всех признаках и на PCA признаках, сравнить метрики


### Kernel PCA
- Найти линейно-неразделимый датасет. При помощи Kernel PCA увеличить размерность (по теореме если у линейно неразделимых данных увеличить размерность, то их можно будет линейно разделить при помощи PCA)
- ~~Функции вычисления ядерной матрицы, для разных ядер~~


In [72]:
import numpy as np
import pandas as pd

Центрирование

In [ ]:
def center(X: np.ndarray):
  return X - np.mean(X, axis=0)

## QR

Одним из эффективных методов вычисления собственных векторов и чисел является QR-алгоритм, основанный на последовательном применении QR-разложения к матрице.

In [ ]:
def gramm_shidt(X):
  m = X.shape[1]
  V = X.copy().astype(np.float64)

  Q = np.zeros_like(V)
  R = np.zeros((m, m))
  for i in range(m):
    R[i, i] = np.linalg.norm(V[:, i])

    if np.isclose(R[i, i], 0, atol=1e-12):
      Q[:, i] = np.random.randn(X.shape[0])
      for k in range(i):
          proj = np.dot(Q[:, k], Q[:, i])
          Q[:, i] = Q[:, i] - proj * Q[:, k]

      Q[:, i] = Q[:, i] / np.linalg.norm(Q[:, i])
      continue

    Q[:, i] = V[:, i] / R[i, i]
    for j in range(i+1, m):
      R[i, j] = np.dot(Q[:, i], V[:, j])
      V[:, j] = V[:, j] - R[i, j]*Q[:, i]

  return Q, R

In [ ]:
# @title
def qr(X, iters=10):
  A = X.copy()
  n, m = X.shape
  Q, R = gramm_shidt(A)
  eig_vectors = np.eye(A.shape[0])

  for i in range(iters):
    Q, R = gramm_shidt(A)
    A = np.dot(R, Q)
    eig_vectors = np.dot(eig_vectors, Q)

  eig_values = np.diag(A)
  return eig_values, eig_vectors

In [ ]:
def shifted_qr(X, iters=10):
  A = X.copy().astype(np.float64)
  n, m = X.shape
  Q, R = (0, 0)
  eig_vectors = np.eye(A.shape[0])

  for i in range(iters):
    Q, R = gramm_shidt(A - A[-1, -1] * np.eye(A.shape[0]))
    A = np.dot(R, Q) + A[-1, -1] * np.eye(A.shape[0])
    eig_vectors = np.dot(eig_vectors, Q)

  eig_values = np.abs(np.diag(A))

  index = np.argsort(eig_values)[::-1]
  eig_values = eig_values[index]
  eig_vectors = eig_vectors[:, index]

  return eig_values, eig_vectors

In [ ]:
X = np.array([[1, 2, 4],
              [-1, 4, 2],
              [1, 2, 3],
              [1, 1, 3]
              ])

eig_values, eig_vectors = shifted_qr(X @ X.T, iters=1000)
value, vector = np.linalg.eig(X @ X.T)
print(np.sort(value)[::-1], eig_values, sep="\n")
print(vector, eig_vectors, sep="\n")

[5.82247819e+01 8.65214592e+00 1.23072206e-01 1.78171847e-15]
[5.82247819e+01 8.65214592e+00 1.23072206e-01 4.26325641e-14]
[[ 5.88154987e-01  3.11509353e-01 -6.35000635e-01  3.92185961e-01]
 [ 5.04391559e-01 -8.45501853e-01  1.27000127e-01  1.20775577e-01]
 [ 4.84535839e-01  1.66378483e-01 -1.30792923e-14 -8.58803366e-01]
 [ 4.06063896e-01  4.00508103e-01  7.62000762e-01  3.06692372e-01]]
[[ 5.88154987e-01  3.11509353e-01 -3.92185839e-01 -6.35000711e-01]
 [ 5.04391559e-01 -8.45501853e-01 -1.20775601e-01  1.27000104e-01]
 [ 4.84535839e-01  1.66378483e-01  8.58803366e-01  1.65426175e-07]
 [ 4.06063896e-01  4.00508103e-01 -3.06692518e-01  7.62000703e-01]]


## Truncated SVD

In [ ]:
def randomized_truncated(A, k, p=5, q_iter=2):
    m, n = A.shape
    l = k + p
    Omega = np.random.randn(n, l)
    Y = A @ Omega

    for _ in range(q_iter):
        Q, _ = gramm_shidt(Y)
        Y = A @ (A.T @ Q)

    Q, _ = gramm_shidt(Y)
    B = Q.T @ A

    eig_vals, eig_vecs = shifted_qr(B @ B.T, iters=100)

    Sigma = np.sqrt(np.maximum(eig_vals, 0))

    V = (B.T @ eig_vecs) / (Sigma + 1e-12)
    U = Q @ eig_vecs

    U = U[:, :k]
    Sigma = Sigma[:k]
    V = V[:, :k]

    return U, Sigma, V


## SVD

Вычисление сингулярных векторов и сингулярных чисел для матрицы X можно
производить следующим образом.

> Для определения сингулярных значений матрицы нужно вычислить собственные значения матрицы XX^T.

> Левые и правые сингулярные векторы определяются через собственные векторы матриц XX^T и X^TX.

In [ ]:
def svd(XXt: np.ndarray, XtX: np.ndarray):
  eigval_U, eigvec_U = shifted_qr(XXt, 100)
  eigval_V, eigvec_V = shifted_qr(XtX, 100)

  singular = np.sqrt(np.abs(eigval_U))
  index = np.argsort(singular)[::-1]
  singular = singular[index]

  index_V = np.argsort(np.sqrt(np.abs(eigval_V)))[::-1]
  eigvec_V = eigvec_V[index_V]

  S = np.zeros((XXt.shape[0], XtX.shape[1]))
  np.fill_diagonal(S, singular)

  return eigvec_U, S, eigvec_V

In [ ]:
np_svd = np.linalg.svd(X)
print(np_svd.U, np_svd.S, np_svd.Vh, sep="\n")

[[-5.88154987e-01  3.11509353e-01  3.92185961e-01 -6.35000635e-01]
 [-5.04391559e-01 -8.45501853e-01  1.20775577e-01  1.27000127e-01]
 [-4.84535839e-01  1.66378483e-01 -8.58803366e-01  2.77555756e-17]
 [-4.06063896e-01  4.00508103e-01  3.06692372e-01  7.62000762e-01]]
[7.63051649 2.94145303 0.35081648]
[[-0.12769295 -0.59878146 -0.79066761]
 [ 0.58607014 -0.68868128  0.42689564]
 [-0.80013518 -0.40887512  0.43886768]]


In [ ]:
U, S, V = svd(X @ X.T, X.T @ X)
print(U, S, V.T, sep="\n")
print(U @ S @ V.T)

[[ 0.58815499 -0.31150935 -0.34251209 -0.6631147 ]
 [ 0.50439156  0.84550185 -0.13012779  0.11739902]
 [ 0.48453584 -0.16637848  0.85629195  0.06563009]
 [ 0.4060639  -0.4005081  -0.36402791  0.73633488]]
[[7.63051649 0.         0.        ]
 [0.         2.94145303 0.        ]
 [0.         0.         0.34979059]
 [0.         0.         0.        ]]
[[-0.12769295 -0.59878146 -0.79066761]
 [-0.58607008  0.68868131 -0.42689567]
 [ 0.80013522  0.40887506 -0.43886764]]
[[-0.13192853 -3.36730525 -3.10471807]
 [-1.9854393  -0.61042877 -4.08481139]
 [ 0.05436439 -2.42841937 -2.84583323]
 [ 0.19289721 -2.718693   -1.8910676 ]]


## PCA

In [ ]:
class MyPCA():
  def __init__(self, n_components) -> None:
    self.n_components = n_components
    self.singular_values  = []
    self.explainded_variance = []

  def fit(self, X, iters=1000, auto_n_components=False, thrs=0.8):
    self.n_components = min(self.n_components, len(X[0]))

    X = center(X)
    eig_values, eig_vectors = shifted_qr(np.dot(X.T, X), iters)
    self.singular_values = np.sqrt(eig_values)

    if auto_n_components:
      singular_s = 0
      total = sum(self.singular_values)
      for index, value in enumerate(self.singular_values):
        singular_s += value
        self.explainded_variance.append(singular_s / total)
        print(f"{index + 1} Компоненты объясняют {self.explainded_variance[-1]:.2f} дисперсии")
        if self.explainded_variance[-1] > thrs:
          return np.dot(X, eig_vectors[:, :index+1])

    self.explainded_variance = [sum(self.singular_values[:index+1]) / sum(self.singular_values) for index in range(self.n_components)]
    return np.dot(X, eig_vectors[:, :self.n_components])

# Kernel PCA

Полиномиальное ядро

In [ ]:
def polynomial(v1, v2, gamma=1, c=0, d=2):
  return (gamma * np.dot(v1, v2) + c) ** d

Радивльное ядро

In [ ]:
def rbf(v1, v2, gamma=1):
  return np.exp(-gamma * np.sum((v1 - v2)**2))

Сигмоидальное ядро

In [ ]:
def sigmoid(v1, v2, gamma=1, c=0):
  return np.tanh(gamma * np.dot(v1, v2) + c)

Подсчет ядерной матрицы

In [ ]:
def K_matrix(X, kernel=polynomial, **kernel_param):
  n = X.shape[0]
  K = np.zeros((n, n))
  for i in range(n):
    for j in range(n):
      K[i, j] = kernel(X[i], X[j], **kernel_param)
  return K

In [ ]:
def center_K(K):
  n = (np.ones_like(K) / K.shape[0])
  return K - np.dot(n, K) - np.dot(K, n) + np.dot(np.dot(n, K), n)

In [ ]:
class MyKernelPCA():
  def __init__(self, n_components) -> None:
    self.n_components = n_components
    self.singular_values  = []
    self.explainded_variance = []

  def fit(self, X):
    self.n_components = min(self.n_components, len(X[0]))
    eig_values, eig_vectors = shifted_qr(X, 100)

    self.explainded_variance = [sum(eig_values[:index+1]) / sum(eig_values) for index in range(len(eig_values))]
    self.singular_values = eig_values

    return np.dot(X, eig_vectors[:, :self.n_components])

# Применение SVD разложения на данные

UΣV^t

Σ - матрица сингулярных чисел показывает дисперсию компоненты

V^t - матрица главных компонент

In [ ]:
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Wine


In [ ]:
from sklearn.datasets import load_wine, make_circles
from sklearn.preprocessing import StandardScaler

In [ ]:
X, y = load_wine(return_X_y=True)
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
data = pd.DataFrame(MyPCA(5).fit(X))
data["claster"] = y

In [ ]:
kpca_data_sigmoid = pd.DataFrame(MyKernelPCA(5).fit(center_K(K_matrix(X, kernel=sigmoid))))
kpca_data_sigmoid["claster"] = y

kpca_data_rbf = pd.DataFrame(MyKernelPCA(5).fit(center_K(K_matrix(X, kernel=rbf, gamma=0.5))))
kpca_data_rbf["claster"] = y

kpca_data_poly = pd.DataFrame(MyKernelPCA(5).fit(center_K(K_matrix(X, kernel=polynomial, d=3))))
kpca_data_poly["claster"] = y

In [ ]:
basic = LogisticRegression()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
basic.fit(X_train, y_train)
print(classification_report(y_test, basic.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        16
           2       1.00      1.00      1.00         6

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [ ]:
pca_model = LogisticRegression()
X_train, X_test, y_train, y_test = train_test_split(data.drop("claster", axis=1).values, y, test_size=0.2, random_state=0)
pca_model.fit(X_train, y_train)
print(classification_report(y_test, pca_model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.93      1.00      0.97        14
           1       1.00      0.94      0.97        16
           2       1.00      1.00      1.00         6

    accuracy                           0.97        36
   macro avg       0.98      0.98      0.98        36
weighted avg       0.97      0.97      0.97        36



In [ ]:
kpca_model = LogisticRegression()
X_train, X_test, y_train, y_test = train_test_split(kpca_data_sigmoid.drop("claster", axis=1).values, y, test_size=0.2, random_state=0)
kpca_model.fit(X_train, y_train)
print(classification_report(y_test, kpca_model.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        16
           2       1.00      1.00      1.00         6

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [ ]:
fig = px.scatter(data_frame=data, x=0, y=1, color="claster")
fig.update_layout(
    title="Проекция на первые 2 компоненты Wine",
    title_font_size=22
)
fig.show()

In [ ]:
# @title
def drawKPCA():
  fig = make_subplots(rows=1, cols=3, column_titles=["Sigmoid", "RBF", "Polynomial"])
  fig.add_trace(
        go.Scatter(
            x=kpca_data_sigmoid[0],
            y=kpca_data_sigmoid[1],
            mode='markers',
            marker=dict(
                color=kpca_data_sigmoid['claster'],
                colorscale='viridis'
            ),
          name='Sigmoid'
        ),
        row=1, col=1
    )
  fig.add_trace(
        go.Scatter(
            x=kpca_data_rbf[0],
            y=kpca_data_rbf[1],
            mode='markers',
            marker=dict(
                color=kpca_data_rbf['claster'],
                colorscale='viridis'
            ),
          name='RBF'
        ),
        row=1, col=2
    )
  fig.add_trace(
        go.Scatter(
            x=kpca_data_poly[0],
            y=kpca_data_poly[1],
            mode='markers',
            marker=dict(
                color=kpca_data_poly['claster'],
                colorscale='viridis'
            ),
          name='Polynomial'
        ),
        row=1, col=3
    )
  fig.update_layout(height=600, width=1920, showlegend=False, title_text="KernelPCA", title_font_size=26, title_x=0.5)
  fig.show()

In [ ]:
drawKPCA()

In [ ]:
X, y = make_circles(n_samples=200, factor=0.3, noise=0.05, random_state=0)

In [ ]:
kpca_data_sigmoid = pd.DataFrame(MyKernelPCA(2).fit(center_K(K_matrix(X, kernel=sigmoid))))
kpca_data_sigmoid["claster"] = y

kpca_data_rbf = pd.DataFrame(MyKernelPCA(2).fit(center_K(K_matrix(X, kernel=rbf, gamma=10))))
kpca_data_rbf["claster"] = y

kpca_data_poly = pd.DataFrame(MyKernelPCA(2).fit(center_K(K_matrix(X, kernel=polynomial, d=3))))
kpca_data_poly["claster"] = y

In [ ]:
drawKPCA()

In [ ]:
circles = LogisticRegression()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
circles.fit(X_train, y_train)
print(classification_report(y_test, circles.predict(X_test)))

              precision    recall  f1-score   support

           0       0.80      0.36      0.50        22
           1       0.53      0.89      0.67        18

    accuracy                           0.60        40
   macro avg       0.67      0.63      0.58        40
weighted avg       0.68      0.60      0.57        40



In [ ]:
rbf_circles = LogisticRegression()
X_train, X_test, y_train, y_test = train_test_split(kpca_data_rbf.drop("claster", axis=1).values, y, test_size=0.2, random_state=0)
rbf_circles.fit(X_train, y_train)
print(classification_report(y_test, rbf_circles.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        22
           1       1.00      1.00      1.00        18

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



# MNIST

In [73]:
from sklearn.preprocessing import StandardScaler

In [74]:
!gdown 1iT7W1VCbBkQuhbmiKkheCq4x8noQmWwF

Downloading...
From: https://drive.google.com/uc?id=1iT7W1VCbBkQuhbmiKkheCq4x8noQmWwF
To: /content/fashion-mnist_test.csv
100% 22.2M/22.2M [00:00<00:00, 66.2MB/s]


In [75]:
mnist = pd.read_csv("/content/fashion-mnist_test.csv")
X, y = mnist.drop("label", axis=1), mnist.label
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
basic = LogisticRegression()
basic.fit(X, y)
print(classification_report(y, basic.predict(X)))

              precision    recall  f1-score   support

           0       0.89      0.91      0.90      1000
           1       1.00      0.99      1.00      1000
           2       0.88      0.87      0.87      1000
           3       0.95      0.95      0.95      1000
           4       0.87      0.89      0.88      1000
           5       0.99      0.98      0.99      1000
           6       0.82      0.79      0.81      1000
           7       0.97      0.98      0.98      1000
           8       1.00      1.00      1.00      1000
           9       0.99      0.99      0.99      1000

    accuracy                           0.94     10000
   macro avg       0.94      0.94      0.94     10000
weighted avg       0.94      0.94      0.94     10000



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



## PCA

In [76]:
U, S, V = randomized_truncated(X, k=150, q_iter=10)

In [77]:
mnist_pca = pd.DataFrame(X @ V)
mnist_pca["claster"] = y

In [78]:
mnist_model = LogisticRegression(max_iter=1000)
mnist_model.fit(mnist_pca.drop("claster", axis=1), y)
print(classification_report(y, mnist_model.predict(mnist_pca.drop("claster", axis=1))))

              precision    recall  f1-score   support

           0       0.74      0.78      0.76      1000
           1       0.96      0.92      0.94      1000
           2       0.64      0.58      0.61      1000
           3       0.75      0.85      0.80      1000
           4       0.59      0.64      0.61      1000
           5       0.84      0.83      0.84      1000
           6       0.42      0.35      0.38      1000
           7       0.81      0.81      0.81      1000
           8       0.91      0.93      0.92      1000
           9       0.87      0.88      0.87      1000

    accuracy                           0.76     10000
   macro avg       0.75      0.76      0.75     10000
weighted avg       0.75      0.76      0.75     10000



In [ ]:
fig = px.scatter(data_frame=mnist_pca, x=0, y=1, color="claster")
fig.update_layout(
    title="Проекция MNIST на первые 2 компоненты",
    title_font_size=22
)
fig.show()

## KernelPCA - Не получается посчитать ядерную матрицу на полной выборке

In [ ]:
K = center_K(K_matrix(X[:1000, :], kernel=polynomial))

In [ ]:
U, S, V = randomized_truncated(K, k=100, q_iter=10)
mnist_kernel_pca = pd.DataFrame(K @ V)
mnist_kernel_pca["claster"] = y[:1000]

In [ ]:
mnist_kpca_model = LogisticRegression()
mnist_kpca_model.fit(mnist_kernel_pca.drop("claster", axis=1), y[:1000])
print(classification_report(y[:1000], mnist_kpca_model.predict(mnist_kernel_pca.drop("claster", axis=1))))

              precision    recall  f1-score   support

           0       0.89      0.92      0.90        98
           1       1.00      0.99      0.99        89
           2       0.84      0.83      0.83       111
           3       0.92      0.93      0.92       104
           4       0.81      0.75      0.78        96
           5       0.98      0.97      0.97        97
           6       0.73      0.75      0.74       100
           7       0.96      0.97      0.97       102
           8       0.98      0.99      0.98        98
           9       0.99      0.99      0.99       105

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



In [ ]:
fig = px.scatter(data_frame=mnist_kernel_pca, x=0, y=1, color="claster")
fig.update_layout(
    title="Ручная реализация",
    title_font_size=22
)
fig.show()

# Данные из 4 лабы

In [ ]:
!gdown 1Ff0ka9Zq9JyyBEgXz1WWIwipRYYBEF_z

Downloading...
From: https://drive.google.com/uc?id=1Ff0ka9Zq9JyyBEgXz1WWIwipRYYBEF_z
To: /content/df_processed.csv
100% 12.3M/12.3M [00:00<00:00, 62.3MB/s]


In [ ]:
bank = pd.read_csv("/content/df_processed.csv")
X, y = bank.drop("Delinquent90", axis=1), " "
pd.DataFrame(MyPCA(2).fit(X.values))

,0,1
0,-0.246920,0.362973
1,0.807940,-1.021225
2,1.430011,1.154045
3,-1.435460,0.633949
4,-1.719405,0.311922
...,...,...
74995,-0.128899,0.718571
74996,-1.147921,0.153616
74997,-1.684911,-0.933221
74998,1.269887,0.017296


In [ ]:
pd.DataFrame(PCA(2).fit_transform(X.values))

,0,1
0,-0.246920,0.362973
1,0.807940,-1.021225
2,1.430011,1.154045
3,-1.435460,0.633949
4,-1.719405,0.311922
...,...,...
74995,-0.128899,0.718571
74996,-1.147921,0.153616
74997,-1.684911,-0.933221
74998,1.269887,0.017296
